# 01 — Разведочный анализ данных (EDA)
**Цель**: изучить структуру временных рядов, паттерны заболеваемости,
связи с внешними факторами, выделить омикрон-период.
> **Автор**: Кибешев Д.М., ИТМО, 2025–2026

## 0. Зависимости

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from utils import smooth_series, mark_omicron, set_plot_style
set_plot_style()
FIGURES = "../results/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Зависимости загружены ✓")

## 1. Загрузка данных
| Источник | Файл | Описание |
|---|---|---|
| Стопкоронавирус.рф / OurWorldInData | `covid_spb.csv` | Суточные случаи, тесты, positive rate |
| Open-Meteo | `weather_spb.csv` | Температура, влажность, осадки |
| Google Community Mobility | `mobility_spb.csv` | Изменение мобильности (%) |
| Яндекс WordStat | `yandex_queries.csv` | Запросы: симптомы, потеря обоняния и т.д. |
| Oxford Stringency Index | `stringency_spb.csv` | Индекс ограничительных мер (0–100) |

In [ ]:
# ─── Реальная загрузка (раскомментируйте когда данные готовы) ───
# df_covid = pd.read_csv("../data/raw/covid_spb.csv", parse_dates=["date"])
# df_covid = df_covid.sort_values("date").reset_index(drop=True)
# df_covid["new_cases_smooth"] = smooth_series(df_covid["new_cases"])
# df_weather = pd.read_csv("../data/raw/weather_spb.csv", parse_dates=["date"])
# df_mobility = pd.read_csv("../data/raw/mobility_spb.csv", parse_dates=["date"])
# df_yandex = pd.read_csv("../data/raw/yandex_queries.csv", parse_dates=["date"])
# df_stringency = pd.read_csv("../data/raw/stringency_spb.csv", parse_dates=["date"])

# ─── ЗАГЛУШКА (замените реальными данными) ───
date_range = pd.date_range("2021-01-01", "2023-06-30", freq="D")
np.random.seed(42)
n = len(date_range)
trend = np.concatenate([
    np.exp(np.linspace(4, 6, 365)),
    np.exp(np.linspace(6, 8, 365)) * 3,
    np.exp(np.linspace(8, 5, n - 730))
])
noise = np.random.normal(0, trend * 0.15)
cases = np.maximum(trend + noise, 0).astype(int)
df_covid = pd.DataFrame({"date": date_range, "new_cases": cases})
df_covid["new_cases_smooth"] = smooth_series(df_covid["new_cases"])
df_covid = mark_omicron(df_covid)
print(df_covid.shape)
df_covid.head()

## 2. Основные характеристики ряда

In [ ]:
print("=== Описательная статистика ===")
print(df_covid["new_cases"].describe().round(2))
print(f"\nПериод: {df_covid['date'].min().date()} → {df_covid['date'].max().date()}")
print(f"Пропуски: {df_covid['new_cases'].isna().sum()}")
print(f"Омикрон-период: {df_covid[df_covid['omicron_wave']==1]['date'].min().date()} →")

## 3. Визуализация временного ряда

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax = axes[0]
ax.fill_between(df_covid["date"], df_covid["new_cases"], alpha=0.3, color="#2980b9", label="Суточные случаи")
ax.plot(df_covid["date"], df_covid["new_cases_smooth"], color="#2c3e50", linewidth=1.5, label="MA-7")
ax.axvspan(df_covid[df_covid["omicron_wave"]==1]["date"].min(),
           df_covid["date"].max(), alpha=0.1, color="#e74c3c", label="Омикрон")
ax.set_ylabel("Новые случаи")
ax.set_title("COVID-19: Суточные случаи (Санкт-Петербург)")
ax.legend()
ax = axes[1]
ax.plot(df_covid["date"], df_covid["new_cases"].pct_change(7)*100,
        color="#8e44ad", linewidth=1, alpha=0.7)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Изменение к пр. неделе (%)")
ax.set_ylim(-200, 200)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{FIGURES}/01_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Недельная сезонность

In [ ]:
df_covid["weekday"] = df_covid["date"].dt.day_name()
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekly = df_covid.groupby("weekday")["new_cases"].mean().reindex(weekday_order)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(7), weekly.values, color="#3498db", edgecolor="white", width=0.7)
ax.set_xticks(range(7))
ax.set_xticklabels(["Пн","Вт","Ср","Чт","Пт","Сб","Вс"])
ax.set_ylabel("Средние суточные случаи")
ax.set_title("Недельная сезонность")
for bar, val in zip(bars, weekly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{val:.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGURES}/01_weekly_pattern.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Сравнение волн: до омикрона vs омикрон

In [ ]:
pre_omicron = df_covid[df_covid["omicron_wave"]==0]["new_cases"]
omicron     = df_covid[df_covid["omicron_wave"]==1]["new_cases"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, label, color in zip(axes,
    [pre_omicron, omicron],
    ["До омикрона (2021)", "Омикрон (2022–2023)"],
    ["#2980b9", "#e74c3c"]):
    ax.hist(data, bins=40, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(data.mean(), color="black", linestyle="--", label=f"Среднее: {data.mean():.0f}")
    ax.set_title(label)
    ax.set_xlabel("Новые случаи/день")
    ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES}/01_wave_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Медиана до омикрона: {pre_omicron.median():.0f}")
print(f"Медиана омикрон:     {omicron.median():.0f}")

## 6. Корреляционный анализ внешних признаков

In [ ]:
# Замените заглушку реальным объединённым датасетом:
# feature_df = df_covid[["date","new_cases_smooth"]]
#              .merge(df_weather, on="date")
#              .merge(df_mobility, on="date")
#              .merge(df_yandex, on="date")
#              .merge(df_stringency, on="date")

np.random.seed(0)
n = len(df_covid)
feature_df = df_covid[["date","new_cases_smooth"]].copy()
feature_df["temperature"]      = np.random.normal(5, 10, n)
feature_df["humidity"]         = np.random.uniform(50, 90, n)
feature_df["transit_mobility"] = np.random.normal(-20, 15, n)
feature_df["stringency"]       = np.random.uniform(0, 80, n)
feature_df["query_symptoms"]   = df_covid["new_cases_smooth"] * np.random.uniform(0.8, 1.2, n)
feature_df["positive_rate"]    = np.random.uniform(0.01, 0.25, n)

corr = feature_df.drop(columns=["date"]).corr()["new_cases_smooth"].drop("new_cases_smooth").sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v > 0 else "#2980b9" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Корреляция Пирсона с new_cases_smooth")
ax.set_title("Корреляция внешних признаков с заболеваемостью")
plt.tight_layout()
plt.savefig(f"{FIGURES}/01_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Итоги EDA
| Наблюдение | Вывод |
|---|---|
| Период данных | 2021-01-01 → 2023-06-30 |
| Ключевой перелом | 01.01.2022 — начало омикрон-волны |
| Недельная сезонность | Спад в выходные (эффект регистрации) |
| Топ коррелирующие признаки | query_symptoms, positive_rate, transit_mobility |

**Следующий шаг** → `02_baseline.ipynb`